# FINTECH545 Practice Midterm 2 — Claude's solution

Every calculation uses Roger's `Library/` package (via `Risk_Library`) wherever it has a matching function.
Where it doesn't, the small extra step is written out inline and marked **(inline: not in Library)**.

At the end of each problem, a **check cell** compares the results with `answerkey.pdf`. Those numbers were also cross-checked against the
answer-generating code in `data generation.jl`.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

from Risk_Library import (
    first_four_moments,
    fit_normal_distribution,
    fit_t_distribution,
    t_distribution_aicc,
    calculate_var,
    calculate_es,
    ew_covariance_matrix,
    classify_psd,
    fix_psd_matrix,
    return_calculate,
    simulate_copula,
    calculate_portfolio_var_es,
)
from utils import aicc   # shared AICc helper (Library/utils.py)

pd.set_option("display.precision", 6)


def normal_aicc(data, params):
    # (inline: not in Library) AICc of a fitted normal = Library's utils.aicc applied to the
    # normal log-likelihood with 2 parameters (mu, sigma). The t version is t_distribution_aicc.
    loglik = stats.norm.logpdf(data, loc=params["mu"], scale=params["sigma"]).sum()
    return aicc(loglik, num_params=2, num_obs=len(data))


def course_historical_var_es(x, alpha):
    # (inline: not in Library) The answer key's empirical-quantile rule (RiskStats.jl):
    # VaR cutoff = average of the order statistics at ceil(n*alpha) and floor(n*alpha);
    # ES = mean of all observations at or below that cutoff. Returned as positive losses.
    # The Library's method="historical" uses np.quantile's linear interpolation instead, which
    # only differs in which neighbouring sorted observations the cutoff is taken from.
    s = np.sort(np.asarray(x, dtype=float))
    n = len(s)
    cutoff = 0.5 * (s[int(np.ceil(n * alpha)) - 1] + s[int(np.floor(n * alpha)) - 1])
    return -cutoff, -s[s <= cutoff].mean()

## Problem 1 (20 pts) — `problem1.csv`
a. Mean, standard deviation, skewness, kurtosis (2)
b. Normal or t? Why? (3)
c. Fit both; prove/disprove (b) (4)
d. 5% and 1% VaR and ES for each distribution (4)
e. 5% and 1% VaR and ES by historical simulation (3)
f. Compare (d) and (e); relate to (b) and (c) (4)

In [2]:
data1 = pd.read_csv("problem1.csv")["X"]

# (a) Library moments.
#  - Std dev: square root of the sample (n-1) variance from method="unbiased".
#  - Skewness / kurtosis: method="biased" (population formulas), the convention the answer key uses.
#    Excess kurtosis: a normal distribution = 0.
m_unbiased = first_four_moments(data1, method="unbiased", kurtosis_type="excess")
m_biased = first_four_moments(data1, method="biased", kurtosis_type="excess")
summary1 = pd.Series({
    "Mean": m_unbiased["Mean"],
    "Std Dev": np.sqrt(m_unbiased["Variance"]),
    "Skewness": m_biased["Skewness"],
    "Kurtosis (excess)": m_biased["Kurtosis"],
})
summary1

Mean                -0.000532
Std Dev              0.031728
Skewness            -0.239510
Kurtosis (excess)    0.680439
dtype: float64

**(b)** Excess kurtosis is ≈ 0.68: positive, so the tails are fatter than a normal distribution's, but only mildly. There is also some negative
skew (≈ −0.24). Because kurtosis is clearly above 0, I would **lean towards the t-distribution**. It is a borderline case, though:
a t with a fairly high ν is very close to a normal, so the normal could plausibly win a fit test.

In [3]:
# (c) Fit both with the Library and compare AICc (lower is better)
normal1 = fit_normal_distribution(data1)          # mu, sigma (sample std)
t1 = fit_t_distribution(data1)                    # MLE mu, sigma, nu
display(pd.DataFrame({"Normal": normal1, "T": t1}))

aicc1 = pd.Series({"Normal AICc": normal_aicc(data1, normal1), "T AICc": t_distribution_aicc(data1, fitted=t1)})
display(aicc1)
print("Better fit:", "T" if aicc1["T AICc"] < aicc1["Normal AICc"] else "Normal")

,Normal,T
mu,-0.000532,-0.000219
nu,NaN,13.711803
sigma,0.031728,0.029299


Normal AICc   -4060.232531
T AICc        -4068.109480
dtype: float64

Better fit: T


**(c)** The t has the lower AICc (≈ −4068.1 vs ≈ −4060.2, a gap of ≈ 8), so the t is the better fit. This **confirms (b)**. The fitted ν ≈ 13.7 is
fairly high, which agrees with the "only mildly fat-tailed" reading from the kurtosis.

In [4]:
# (d) Parametric VaR / ES (as positive losses measured from 0) for each fitted distribution, closed form
# (e) Historical simulation: no distribution, just the empirical tail of the observed returns (Library method="historical")
rows = {}
for alpha in (0.05, 0.01):
    for dist in ("normal", "t"):
        rows[(f"{dist.capitalize()}", f"{alpha:.0%}")] = {
            "VaR": calculate_var(data1, distribution=dist, method="analytical", alpha=alpha)["VaR Absolute"],
            "ES": calculate_es(data1, distribution=dist, method="analytical", alpha=alpha)["ES Absolute"],
        }
    rows[("Historical (Library)", f"{alpha:.0%}")] = {
        "VaR": calculate_var(data1, method="historical", alpha=alpha)["VaR Absolute"],
        "ES": calculate_es(data1, method="historical", alpha=alpha)["ES Absolute"],
    }
    v, e = course_historical_var_es(data1, alpha)
    rows[("Historical (answer-key rule)", f"{alpha:.0%}")] = {"VaR": v, "ES": e}

risk1 = pd.DataFrame(rows).T
risk1.index.names = ["Method", "alpha"]
risk1 = risk1.sort_index(level=["alpha", "Method"], ascending=[False, True])
risk1

,,VaR,ES
Method,alpha,,
Historical (Library),5%,0.053356,0.071073
Historical (answer-key rule),5%,0.053655,0.071073
Normal,5%,0.052719,0.065977
T,5%,0.051901,0.067714
Historical (Library),1%,0.075974,0.099050
Historical (answer-key rule),1%,0.077373,0.099050
Normal,1%,0.074342,0.085093
T,1%,0.077320,0.092326


Note on (e): the two historical rows use the same data and differ only in how the empirical quantile is picked between neighbouring sorted
observations. The Library uses `np.quantile`'s linear interpolation. The answer key averages the ⌊nα⌋-th and ⌈nα⌉-th smallest values. Historical ES is
identical in both at 5% and 1% here. Historical VaR differs by about 0.0003 at 5% and 0.0014 at 1%. That gap shows how noisy a tail estimate built from
only 10–50 points is.

**(f) Comparison.**
- At **5%**, both parametric models give a *smaller* VaR and ES than history (VaR ≈ 0.0527 / 0.0519 vs ≈ 0.0537; ES ≈ 0.066 / 0.068 vs ≈ 0.071).
- At **1%**, the normal clearly understates risk (VaR 0.074, ES 0.085). The t is closer: its VaR (0.0773) almost equals the historical 0.0774 (key rule) /
  0.0760 (Library), but its ES (0.092) is still below the historical ES (0.099).
- So the t is the better of the two, which agrees with (b) and (c): fatter tails move it towards the data, especially for ES and deep in the tail.
  But **neither model is perfect**. Both are symmetric, while the data has **negative skew** (≈ −0.24), so the real left tail is longer than either
  symmetric model allows. That is why both underestimate the historical ES. A skewed distribution would be needed to capture it.

In [5]:
# Check against answerkey.pdf
key1 = {
    ("Mean",): -0.0005316702326786278, ("Std Dev",): 0.03172783789203951,
    ("Skewness",): -0.2395102766006735, ("Kurtosis (excess)",): 0.680438623588139,
}
check1 = pd.DataFrame({
    "mine": [*summary1.values, aicc1["Normal AICc"], aicc1["T AICc"], t1["nu"],
             *risk1.loc[[("Normal", "5%"), ("Normal", "1%"), ("T", "5%"), ("T", "1%"),
                         ("Historical (answer-key rule)", "5%"), ("Historical (answer-key rule)", "1%")]].values.ravel()],
    "key": [-0.0005316702326786278, 0.03172783789203951, -0.2395102766006735, 0.680438623588139,
            -4060.2325305235076, -4068.109480219482, 13.711855137418015,
            0.05271931946472819, 0.06597708780254659, 0.07434165846073733, 0.08509315494056131,
            0.05190076384436333, 0.06771378728187447, 0.07731990069879922, 0.092325522076692,
            0.05365484666259305, 0.07107272387024315, 0.07737344675790993, 0.09905041150582906],
}, index=["Mean", "Std", "Skew", "Kurt", "Normal AICc", "T AICc", "T nu",
          "N VaR5", "N ES5", "N VaR1", "N ES1", "T VaR5", "T ES5", "T VaR1", "T ES1",
          "H VaR5", "H ES5", "H VaR1", "H ES1"])
check1["match"] = np.isclose(check1["mine"], check1["key"], rtol=1e-5)
check1

,mine,key,match
Mean,-0.000532,-0.000532,True
Std,0.031728,0.031728,True
Skew,-0.239510,-0.239510,True
Kurt,0.680439,0.680439,True
Normal AICc,-4060.232531,-4060.232531,True
T AICc,-4068.109480,-4068.109480,True
T nu,13.711803,13.711855,True
N VaR5,0.052719,0.052719,True
N ES5,0.065977,0.065977,True
N VaR1,0.074342,0.074342,True


## Problem 2 (20 pts) — `problem2.csv`
Correlations move slowly; variances update faster.
a. Higher or lower λ for the correlation estimator? (5)
b. EW correlation (λ ∈ {0.94, 0.97}) (5)
c. EW variance (5)
d. Combine into a covariance matrix (5)

**(a)** Choose the **higher λ (0.97) for correlation**. In an EW estimator the weight on an observation k periods ago is ∝ λᵏ.
A higher λ decays more slowly, so it keeps a longer memory and spreads weight more evenly across history. That suits a slow-moving
quantity like correlation, and it makes the correlation estimate less noisy. Variances update faster, so they get the **lower λ (0.94)**. That
weights recent data more and lets the volatility estimate react quickly.

In [6]:
data2 = pd.read_csv("problem2.csv")   # oldest observation first, as the Library expects

# (b) EW correlation with the slower decay, lambda = 0.97
ew_corr2 = ew_covariance_matrix(data2, lambda_=0.97, method="correlation")
ew_corr2

,x1,x2,x3,x4,x5
x1,1.000000,0.501728,-0.276033,-0.153536,-0.287344
x2,0.501728,1.000000,0.312724,-0.321841,-0.222609
x3,-0.276033,0.312724,1.000000,-0.073139,0.418567
x4,-0.153536,-0.321841,-0.073139,1.000000,-0.406487
x5,-0.287344,-0.222609,0.418567,-0.406487,1.000000


In [7]:
# (c) EW variances with the faster decay, lambda = 0.94 (diagonal of the lambda=0.94 EW covariance)
ew_var2 = pd.Series(np.diag(ew_covariance_matrix(data2, lambda_=0.94)), index=data2.columns)
pd.DataFrame({"EW variance (λ=0.94)": ew_var2, "EW std dev": np.sqrt(ew_var2)})

,EW variance (λ=0.94),EW std dev
x1,0.000353,0.018798
x2,0.000555,0.023562
x3,0.000187,0.013658
x4,0.000420,0.020493
x5,0.000126,0.011225


In [8]:
# (d) Cov = diag(sd_0.94) @ Corr_0.97 @ diag(sd_0.94). One Library call:
#     lambda_ = decay for the variances, lambda_corr = decay for the correlation
cov2 = ew_covariance_matrix(data2, lambda_=0.94, method="covariance", lambda_corr=0.97)

sd2 = np.sqrt(ew_var2.values)   # sanity check: same thing assembled by hand from (b) and (c)
assert np.allclose(cov2.values, np.outer(sd2, sd2) * ew_corr2.values)
cov2

,x1,x2,x3,x4,x5
x1,0.000353,0.000222,-0.000071,-0.000059,-0.000061
x2,0.000222,0.000555,0.000101,-0.000155,-0.000059
x3,-0.000071,0.000101,0.000187,-0.000020,0.000064
x4,-0.000059,-0.000155,-0.000020,0.000420,-0.000094
x5,-0.000061,-0.000059,0.000064,-0.000094,0.000126


In [9]:
# Check against answerkey.pdf
key2_corr = np.array([[1.0, 0.501728, -0.276033, -0.153536, -0.287344],
                      [0.501728, 1.0, 0.312724, -0.321841, -0.222609],
                      [-0.276033, 0.312724, 1.0, -0.0731389, 0.418567],
                      [-0.153536, -0.321841, -0.0731389, 1.0, -0.406487],
                      [-0.287344, -0.222609, 0.418567, -0.406487, 1.0]])
key2_var = np.array([0.000353376, 0.000555158, 0.000186538, 0.000419965, 0.000126011])
key2_cov = np.array([[0.000353376, 0.000222226, -7.08703e-5, -5.91472e-5, -6.06351e-5],
                     [0.000222226, 0.000555158, 0.000100636, -0.000155402, -5.88782e-5],
                     [-7.08703e-5, 0.000100636, 0.000186538, -2.0471e-5, 6.41731e-5],
                     [-5.91472e-5, -0.000155402, -2.0471e-5, 0.000419965, -9.351e-5],
                     [-6.06351e-5, -5.88782e-5, 6.41731e-5, -9.351e-5, 0.000126011]])
print("corr match:", np.allclose(ew_corr2, key2_corr, atol=1e-6))
print("var  match:", np.allclose(ew_var2, key2_var, rtol=1e-5))
print("cov  match:", np.allclose(cov2, key2_cov, rtol=1e-5, atol=1e-10))

corr match: True
var  match: True
cov  match: True


## Problem 3 (20 pts) — `problem3.csv` (a given covariance matrix)
a. Eigenvalues: what do you see? (5)
b. Higham nearest PSD; effect on eigenvalues (5)
c. Rebonato–Jäckel near-PSD; effect on eigenvalues (5)
d. Compare; which do you prefer? (5)

In [10]:
cov3 = pd.read_csv("problem3.csv")
cov3.index = cov3.columns

# (a) Eigenvalues (smallest -> largest) plus the Library's PD / PSD / non-PSD diagnosis
eig_orig = np.linalg.eigvalsh(cov3.values)
print("Eigenvalues:", eig_orig)
print(classify_psd(cov3)["summary"])

Eigenvalues: [-4.56160014e-05  1.23566772e-04  1.90829637e-04  4.26728160e-04
  8.64915610e-04]
non-PSD: 1 eigenvalue(s) below -threshold (9.65e-12): [-4.562e-05].


**(a)** One eigenvalue is clearly **negative** (≈ −4.56e-5). It is about a third of the size of the next one, not a rounding-error-sized number.
So the matrix is **not PSD**, which means it is not a valid covariance matrix. Some portfolio `w` would get `wᵀΣw < 0`, a negative variance.
Cholesky would fail, and simulation or VaR built on it would be meaningless. It has to be repaired before use.

In [11]:
# (b) Higham's alternating projections: the nearest PSD matrix (Frobenius norm). For a covariance input
# the Library works on the implied correlation matrix and rescales back, so the variances (diagonal) are kept.
higham3 = fix_psd_matrix(cov3, method="higham")
eig_higham = np.linalg.eigvalsh(higham3.values)
display(higham3)
print("Eigenvalues:", eig_higham)

,x1,x2,x3,x4,x5
x1,0.000338,0.000346,-0.000068,-0.000043,-0.000036
x2,0.000346,0.000501,0.000002,-0.000162,0.000055
x3,-0.000068,0.000002,0.000189,-0.000101,-0.000046
x4,-0.000043,-0.000162,-0.000101,0.000403,-0.000029
x5,-0.000036,0.000055,-0.000046,-0.000029,0.000129


Eigenvalues: [-1.13675738e-13  1.18538629e-04  1.87796512e-04  4.21816738e-04
  8.32272298e-04]


**(b)** Higham pushes the negative eigenvalue up to **0** (≈ 1e-13, numerical zero). The other eigenvalues move only a little and all
shrink slightly (e.g. the largest goes from 8.65e-4 to 8.32e-4). The diagonal (the variances) is unchanged.

In [12]:
# (c) Rebonato & Jackel near-PSD: clip negative eigenvalues of the correlation matrix to 0 (epsilon=0),
# rescale rows back to a unit diagonal, then rescale to covariance with the original variances.
nearpsd3 = fix_psd_matrix(cov3, method="near_psd", epsilon=0.0)
eig_near = np.linalg.eigvalsh(nearpsd3.values)
display(nearpsd3)
print("Eigenvalues:", eig_near)

,x1,x2,x3,x4,x5
x1,0.000338,0.000340,-0.000068,-0.000041,-0.000038
x2,0.000340,0.000501,0.000006,-0.000158,0.000056
x3,-0.000068,0.000006,0.000189,-0.000100,-0.000046
x4,-0.000041,-0.000158,-0.000100,0.000403,-0.000029
x5,-0.000038,0.000056,-0.000046,-0.000029,0.000129


Eigenvalues: [9.29134759e-21 1.24783385e-04 1.88583491e-04 4.22352328e-04
 8.24704974e-04]


**(c)** Near-PSD also moves the negative eigenvalue to **0** (≈ 1e-20) and keeps the variances. Its other eigenvalues change by a
different pattern: the second-smallest actually *rises* (1.236e-4 → 1.248e-4), and the largest falls slightly more than under Higham (→ 8.25e-4).

In [13]:
# (d) Compare how far each repaired matrix is from the original (inline: simple numpy distances)
compare3 = pd.DataFrame({"Original": eig_orig, "Higham": eig_higham, "Near PSD": eig_near})
display(compare3)

diff_h = higham3.values - cov3.values
diff_n = nearpsd3.values - cov3.values
distances3 = pd.DataFrame({
    "Higham": {"Frobenius norm ||A - C||_F": np.linalg.norm(diff_h, "fro"),
               "Sum |A - C|": np.abs(diff_h).sum(),
               "Sum sq. eigenvalue change": ((eig_higham - eig_orig) ** 2).sum()},
    "Near PSD": {"Frobenius norm ||A - C||_F": np.linalg.norm(diff_n, "fro"),
                 "Sum |A - C|": np.abs(diff_n).sum(),
                 "Sum sq. eigenvalue change": ((eig_near - eig_orig) ** 2).sum()},
})
distances3

,Original,Higham,Near PSD
0,-0.000046,-1.136757e-13,9.291348e-21
1,0.000124,1.185386e-04,1.247834e-04
2,0.000191,1.877965e-04,1.885835e-04
3,0.000427,4.218167e-04,4.223523e-04
4,0.000865,8.322723e-04,8.247050e-04


,Higham,Near PSD
Frobenius norm ||A - C||_F,6.227773e-05,6.762283e-05
Sum |A - C|,1.723995e-04,1.797148e-04
Sum sq. eigenvalue change,3.205009e-09,3.723388e-09


**(d)** Both repairs give a valid PSD matrix with the original variances. **Higham is closer to the original matrix** on every measure above:
smaller Frobenius distance, smaller total absolute change, and smaller eigenvalue change. That is expected, because Higham solves for the
*nearest* PSD matrix, while near-PSD is a one-shot heuristic with no optimality guarantee. Near-PSD's advantage is speed (one eigendecomposition
vs many iterations), which only matters for very large matrices. For a 5×5 matrix the cost is negligible, so I prefer **Higham**: it changes the
input as little as possible.

In [14]:
# Check against answerkey.pdf
key3_higham = np.array([[0.000337588, 0.000345672, -6.7811e-5, -4.28635e-5, -3.62003e-5],
                        [0.000345672, 0.000501453, 2.01492e-6, -0.00016224, 5.46521e-5],
                        [-6.7811e-5, 2.01492e-6, 0.000189133, -0.000101158, -4.63366e-5],
                        [-4.28635e-5, -0.00016224, -0.000101158, 0.000403089, -2.92798e-5],
                        [-3.62003e-5, 5.46521e-5, -4.63366e-5, -2.92798e-5, 0.000129162]])
key3_near = np.array([[0.000337588, 0.000339766, -6.84556e-5, -4.11224e-5, -3.79551e-5],
                      [0.000339766, 0.000501453, 5.53656e-6, -0.000158481, 5.64025e-5],
                      [-6.84556e-5, 5.53656e-6, 0.000189133, -0.000100357, -4.64565e-5],
                      [-4.11224e-5, -0.000158481, -0.000100357, 0.000403089, -2.88686e-5],
                      [-3.79551e-5, 5.64025e-5, -4.64565e-5, -2.88686e-5, 0.000129162]])
print("original eigenvalues match:", np.allclose(eig_orig, [-4.5616001351745625e-5, 0.00012356677195028497,
      0.0001908296373330973, 0.0004267281595771177, 0.0008649156097302901], rtol=1e-8))
print("Higham matrix match:      ", np.allclose(higham3, key3_higham, rtol=1e-4, atol=1e-10))
print("Near PSD matrix match:    ", np.allclose(nearpsd3, key3_near, rtol=1e-4, atol=1e-10))
print("eig-distance match (key 3.2050e-9 / 3.7234e-9):",
      np.isclose(distances3.loc["Sum sq. eigenvalue change", "Higham"], 3.2050094492832762e-9, rtol=1e-4),
      np.isclose(distances3.loc["Sum sq. eigenvalue change", "Near PSD"], 3.7233880377460915e-9, rtol=1e-4))
print("sum-abs match (key 1.72400e-4 / 1.79715e-4):",
      np.isclose(distances3.loc["Sum |A - C|", "Higham"], 0.00017239954909766157, rtol=1e-4),
      np.isclose(distances3.loc["Sum |A - C|", "Near PSD"], 0.00017971478760808653, rtol=1e-4))

original eigenvalues match: True
Higham matrix match:       True
Near PSD matrix match:     True
eig-distance match (key 3.2050e-9 / 3.7234e-9): True True
sum-abs match (key 1.72400e-4 / 1.79715e-4): True True


## Problem 4 (60 pts) — `problem4.csv` prices, $100,000 equal-weight portfolio
a. Shares per stock for equal weights (5)
b. Arithmetic daily returns; first/last 5 rows (10)
c. De-mean; fit normal and t per stock; show the best fit (15)
d. 1% VaR and ES ($) per stock and portfolio: Gaussian copula and historical simulation; compare; preference (30)

In [15]:
prices4 = pd.read_csv("problem4.csv")
tickers = [c for c in prices4.columns if c != "Date"]

# (a) Equal weight: each stock gets $100,000 / 5 = $20,000 at today's (last) price  (inline: simple arithmetic)
PORTFOLIO_VALUE = 100_000
current_prices4 = prices4[tickers].iloc[-1]
shares4 = (PORTFOLIO_VALUE / len(tickers)) / current_prices4
position_value4 = shares4 * current_prices4        # $20,000 each
pd.DataFrame({"Current Price": current_prices4, "Shares": shares4, "Position Value ($)": position_value4})

,Current Price,Shares,Position Value ($)
SPY,689.429993,29.009472,20000.0
AAPL,264.579987,75.591507,20000.0
MSFT,397.230011,50.348663,20000.0
GOOGL,314.980011,63.496093,20000.0
BABA,154.449997,129.491747,20000.0


In [16]:
# (b) Arithmetic returns R_t = P_t / P_{t-1} - 1 with the Library (Date column carried along)
returns4 = return_calculate(prices4, method="arithmetic", date_column="Date")
print("First 5 rows:")
display(returns4.head(5))
print("Last 5 rows:")
display(returns4.tail(5))

First 5 rows:


,Date,SPY,AAPL,MSFT,GOOGL,BABA
0,2022-01-04,-0.000335,-0.012692,-0.017147,-0.004083,-0.006812
1,2022-01-05,-0.019202,-0.026600,-0.038388,-0.045876,0.013382
2,2022-01-06,-0.000940,-0.016693,-0.007902,-0.000200,0.045147
3,2022-01-07,-0.003953,0.000988,0.000510,-0.005303,0.025113
4,2022-01-10,-0.001244,0.000116,0.000733,0.012060,-0.011632


Last 5 rows:


,Date,SPY,AAPL,MSFT,GOOGL,BABA
1031,2026-02-13,0.000705,-0.022733,-0.001294,-0.010615,-0.018900
1032,2026-02-17,0.001613,0.031668,-0.011113,-0.012103,-0.001926
1033,2026-02-18,0.005038,0.001781,0.006904,0.004337,0.002188
1034,2026-02-19,-0.002637,-0.014261,-0.002853,-0.001582,-0.009630
1035,2026-02-20,0.007232,0.015350,-0.003087,0.040053,0.001167


In [17]:
# (c) De-mean every series, then fit a normal and a t to each stock and keep the lower-AICc model
returns4 = returns4.set_index("Date")[tickers]
returns4 = returns4 - returns4.mean()

fit_rows = {}
best_dist4 = {}
for tk in tickers:
    n_fit = fit_normal_distribution(returns4[tk])
    t_fit = fit_t_distribution(returns4[tk])
    n_aicc = normal_aicc(returns4[tk], n_fit)
    t_aicc = t_distribution_aicc(returns4[tk], fitted=t_fit)
    best = "T" if t_aicc < n_aicc else "Normal"
    best_dist4[tk] = best
    fit_rows[tk] = {
        "Normal mu": n_fit["mu"], "Normal sigma": n_fit["sigma"], "Normal AICc": n_aicc,
        "T mu": t_fit["mu"], "T sigma": t_fit["sigma"], "T nu": t_fit["nu"], "T AICc": t_aicc,
        "Best fit": best,
    }
fits4 = pd.DataFrame(fit_rows).T
fits4

,Normal mu,Normal sigma,Normal AICc,T mu,T sigma,T nu,T AICc,Best fit
SPY,0.0,0.011237,-6357.204887,0.000279,0.007723,3.664941,-6534.87862,T
AAPL,-0.0,0.017949,-5386.828165,0.000169,0.01219,3.492912,-5558.526787,T
MSFT,0.0,0.017164,-5479.496695,0.000206,0.012529,4.126322,-5601.411553,T
GOOGL,0.0,0.020395,-5122.180854,0.000035,0.015416,4.552051,-5222.750556,T
BABA,0.0,0.032979,-4126.397013,-0.00216,0.02077,3.125143,-4377.449602,T


**(c)** The **t-distribution wins on AICc for every stock**, by a wide margin (≈ 100–250 AICc points). All fitted ν are between ≈ 3.1 and 4.6,
which means very fat tails, typical for daily equity returns. BABA has the fattest tails (ν ≈ 3.1) and the largest scale. The best-fit parameters are the `T mu`,
`T sigma`, and `T nu` columns. The normal μ is exactly 0 because the series were de-meaned. The t location μ is a robust centre, so it is not exactly 0.

In [18]:
# (d-1) Gaussian copula simulation with the Library:
#   - marginals = each stock's best-fit model from (c) (all "t" here),
#   - copula correlation = Spearman rank correlation (same as the answer key),
#   - correlated normals -> uniforms -> each stock's own inverse t CDF -> simulated returns.
NUM_SIMS = 1_000_000
sim_returns4, copula_fit4 = simulate_copula(
    returns4, copula="gaussian",
    marginals={tk: best_dist4[tk].lower() for tk in tickers},
    correlation="spearman", num_simulations=NUM_SIMS, seed=545, output="both",
)
print("Gaussian copula correlation (Spearman):")
copula_fit4["correlation"]

Gaussian copula correlation (Spearman):


,SPY,AAPL,MSFT,GOOGL,BABA
SPY,1.000000,0.695532,0.747113,0.678570,0.373796
AAPL,0.695532,1.000000,0.578527,0.564982,0.310127
MSFT,0.747113,0.578527,1.000000,0.631735,0.257240
GOOGL,0.678570,0.564982,0.631735,1.000000,0.323606
BABA,0.373796,0.310127,0.257240,0.323606,1.000000


In [19]:
# (d-2) $ P&L per position = position value * simulated return; portfolio P&L = sum across stocks  (inline: not in Library)
sim_pnl4 = sim_returns4 * position_value4
sim_pnl4["Total"] = sim_pnl4.sum(axis=1)

# Historical simulation: apply each de-meaned historical return day to today's positions
hist_pnl4 = returns4 * position_value4
hist_pnl4["Total"] = hist_pnl4.sum(axis=1)

# VaR / ES at alpha = 1% of each P&L sample. Library calculate_var/es with method="historical" = empirical
# quantile / tail mean of whatever sample it is given (simulated or historical).
ALPHA = 0.01
results4 = {}
for col in sim_pnl4.columns:
    hv, he = course_historical_var_es(hist_pnl4[col], ALPHA)
    results4[col] = {
        "Copula VaR99": calculate_var(sim_pnl4[col], method="historical", alpha=ALPHA)["VaR Absolute"],
        "Copula ES99": calculate_es(sim_pnl4[col], method="historical", alpha=ALPHA)["ES Absolute"],
        "Hist VaR99 (Library)": calculate_var(hist_pnl4[col], method="historical", alpha=ALPHA)["VaR Absolute"],
        "Hist ES99 (Library)": calculate_es(hist_pnl4[col], method="historical", alpha=ALPHA)["ES Absolute"],
        "Hist VaR99 (key rule)": hv,
        "Hist ES99 (key rule)": he,
    }
risk4 = pd.DataFrame(results4).T
risk4.round(2)

,Copula VaR99,Copula ES99,Hist VaR99 (Library),Hist ES99 (Library),Hist VaR99 (key rule),Hist ES99 (key rule)
SPY,602.64,875.43,634.26,818.26,660.55,835.11
AAPL,981.35,1439.33,973.15,1162.78,977.31,1181.64
MSFT,918.65,1272.64,869.98,1156.32,879.86,1184.15
GOOGL,1082.82,1462.41,1003.99,1388.63,1018.10,1426.58
BABA,1868.30,2809.42,1628.22,1989.55,1673.57,2024.36
Total,3996.63,5371.54,3788.32,4621.07,3826.89,4700.98


In [20]:
# Cross-check: the Library's one-call Gaussian-copula portfolio function (T marginals). It builds the copula
# correlation from normal scores instead of Spearman and uses its own draws, so expect close but not identical numbers.
portfolio4 = pd.DataFrame({"Stock": tickers, "Holding": shares4.values,
                           "Starting Price": current_prices4.values,
                           "Distribution": [best_dist4[tk] for tk in tickers]})
calculate_portfolio_var_es(portfolio4, returns4, alpha=ALPHA, num_simulations=NUM_SIMS, seed=545).round(2)

,Stock,VaR99,ES99,VaR99_Pct,ES99_Pct
0,SPY,602.64,875.43,0.03,0.04
1,AAPL,981.29,1441.75,0.05,0.07
2,MSFT,918.58,1273.00,0.05,0.06
3,GOOGL,1082.62,1462.22,0.05,0.07
4,BABA,1868.75,2809.39,0.09,0.14
5,Total,4023.88,5423.98,0.04,0.05


In [21]:
# Skewness of each de-meaned return series (Library), used in the discussion below
first_four_moments(returns4, method="biased").loc[["Skewness", "Kurtosis"]]

,SPY,AAPL,MSFT,GOOGL,BABA
Skewness,0.357989,0.529644,0.042670,0.088766,1.729328
Kurtosis,9.208256,7.203367,3.963081,2.962366,16.392871


**(d) Comparison and preference.**

| | Copula (Gaussian copula + t marginals) | Historical simulation |
|---|---|---|
| Single stocks | VaR slightly above history for AAPL, MSFT, GOOGL, BABA (SPY slightly below) | — |
| ES | **noticeably higher than history**, most of all for BABA (≈ \$2.8k vs ≈ \$2.0k) | — |
| Portfolio | VaR ≈ \$4.0k, ES ≈ \$5.4k | VaR ≈ \$3.8k, ES ≈ \$4.6–4.7k |

Why they differ:
1. **Tail sample size.** With ≈1036 days, the 1% tail of the historical sample is only ~10 observations. Historical VaR is close to "the 10th-worst
   day", and ES averages only those ~10 days. That is a noisy estimate, and it can never go beyond the worst day observed. The
   fitted t marginals (ν ≈ 3–4) extend the tail past anything in the sample, so the model ES is larger.
2. **Skewness.** The t marginals are symmetric. BABA's de-meaned returns are strongly **positively** skewed (≈ 1.7): its big moves were mostly
   *up*. Fitting a symmetric t to those big moves puts equally large moves in the loss tail, so the model overstates BABA's downside compared with
   history. That is where the biggest gap is.
3. **Dependence.** The Gaussian copula has no tail dependence. In real crashes, stocks tend to fall together more than a Gaussian copula implies, which
   pushes the other way (the model could understate the portfolio tail). Here the marginal effects in points 1 and 2 dominate.

**Preference:** I would rely on **historical simulation** for this portfolio, and keep the copula as a sanity check. History has about four years of
data, it keeps the real skewness of each stock and the real joint behaviour in bad days, and the symmetric t model clearly mis-models BABA's skew. The copula
model is more useful when history is short or when we want to extend beyond observed extremes. For a conservative capital number, the copula ES is a reasonable upper bound.

In [22]:
# Check against answerkey.pdf
# Copula numbers are Monte Carlo: they should lie inside the key's 95% CI (1000 runs x 1000 sims).
key4_shares = pd.Series([29.0095, 75.5915, 50.3487, 63.4961, 129.492], index=tickers)
key4_t = pd.DataFrame({"mu": [0.00027858287499741804, 0.00016870427558700871, 0.0002061935633905086, 3.521150098278354e-5, -0.0021603923076874035],
                       "sigma": [0.007722799294132536, 0.012190250060417279, 0.012529025663853928, 0.015416426707686602, 0.020769639158760393],
                       "nu": [3.6648954473348256, 3.492939841648696, 4.126273032944454, 4.5520248353202035, 3.1251949227351306]}, index=tickers)
print("shares match:     ", np.allclose(shares4, key4_shares, rtol=1e-5))
raw_returns4 = return_calculate(prices4, method="arithmetic", date_column="Date").set_index("Date")[tickers]
print("first/last rows: ", np.allclose(raw_returns4.iloc[0], [-0.000335043, -0.0126915, -0.0171469, -0.00408296, -0.00681181], rtol=1e-5),
      np.allclose(raw_returns4.iloc[-1], [0.00723179, 0.0153504, -0.00308684, 0.0400528, 0.00116674], rtol=1e-5))
print("t fits match:     ", np.allclose(fits4[["T mu", "T sigma", "T nu"]].astype(float).values, key4_t.values, rtol=2e-3, atol=2e-6))

idx = tickers + ["Total"]
key4 = pd.DataFrame({
    "VaR lo": [505.999, 820.611, 777.128, 920.627, 1520.34, 3395.64],
    "VaR hi": [768.556, 1270.84, 1141.26, 1316.9, 2371.14, 4802.64],
    "ES lo":  [637.494, 1055.54, 980.742, 1140.7, 1980.25, 4144.49],
    "ES hi":  [1193.65, 2026.89, 1670.82, 1888.71, 4135.78, 6974.91],
    "Key Hist VaR99": [660.546, 977.308, 879.863, 1018.1, 1673.57, 3826.89],
    "Key Hist ES99":  [835.112, 1181.64, 1184.15, 1426.58, 2024.36, 4700.98],
}, index=idx)
key4["My Copula VaR99"] = risk4["Copula VaR99"]
key4["My Copula ES99"] = risk4["Copula ES99"]
key4["VaR in CI"] = key4["My Copula VaR99"].between(key4["VaR lo"], key4["VaR hi"])
key4["ES in CI"] = key4["My Copula ES99"].between(key4["ES lo"], key4["ES hi"])
key4["Hist match (key rule)"] = (np.isclose(risk4["Hist VaR99 (key rule)"], key4["Key Hist VaR99"], rtol=1e-5)
                                 & np.isclose(risk4["Hist ES99 (key rule)"], key4["Key Hist ES99"], rtol=1e-5))
key4.round(2)

shares match:      True
first/last rows:  True True
t fits match:      True


,VaR lo,VaR hi,ES lo,ES hi,Key Hist VaR99,Key Hist ES99,My Copula VaR99,My Copula ES99,VaR in CI,ES in CI,Hist match (key rule)
SPY,506.00,768.56,637.49,1193.65,660.55,835.11,602.64,875.43,True,True,True
AAPL,820.61,1270.84,1055.54,2026.89,977.31,1181.64,981.35,1439.33,True,True,True
MSFT,777.13,1141.26,980.74,1670.82,879.86,1184.15,918.65,1272.64,True,True,True
GOOGL,920.63,1316.90,1140.70,1888.71,1018.10,1426.58,1082.82,1462.41,True,True,True
BABA,1520.34,2371.14,1980.25,4135.78,1673.57,2024.36,1868.30,2809.42,True,True,True
Total,3395.64,4802.64,4144.49,6974.91,3826.89,4700.98,3996.63,5371.54,True,True,True
